# Customer Segmentation Pipeline — Walkthrough
**Prodigy Infotech · Task-02**

This notebook is a transparent, narrative walkthrough of the enterprise K-means clustering pipeline built in `src/`. Every step that `scripts/run_pipeline.py` performs automatically is reproduced here with explanation, so you can see *why* each decision was made, not just *what* the code does.

**Dataset:** [Mall Customers (Kaggle)](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)  
**Algorithm:** K-means (k=5, `k-means++` initialisation)  
**Features:** Age · Annual Income (k\$) · Spending Score (1-100)

---
### Table of Contents
1. Environment Setup  
2. Configuration Loading  
3. Data Loading & Validation  
4. Exploratory Data Analysis  
5. Feature Engineering  
6. Model Training  
7. Cluster Evaluation  
8. Visualisations  
9. Business Interpretation  
10. Saving Artifacts


## 1. Environment Setup

We add the project root to `sys.path` so notebook imports resolve the same way the pipeline script does. All logging is routed through the same structured logger used in production — no raw `print()` calls.

In [ ]:
import sys
import warnings
from pathlib import Path

# Make project root importable regardless of where the kernel is launched from
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.utils.config_loader import load_config
from src.utils.logger import get_logger
from src.data.loader import load_raw_data
from src.features.feature_engineering import FeatureEngineer
from src.models.kmeans_model import CustomerSegmentationModel
from src.models.evaluation import (
    compute_silhouette, compute_davies_bouldin,
    compute_elbow_curve, build_cluster_profiles,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy {np.__version__} | Pandas {pd.__version__}")


## 2. Configuration Loading

All tunable parameters live in `config/config.yaml`. The loader validates required sections and rewrites relative paths to absolute ones anchored at the project root, so the pipeline behaves identically regardless of which directory it is invoked from.

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "config.yaml")

print("Project         :", config["project"]["name"], "v" + config["project"]["version"])
print("n_clusters      :", config["model"]["n_clusters"])
print("Clustering feats:", config["features"]["clustering_features"])
print("Random seed     :", config["project"]["random_seed"])
print("Raw data path   :", config["data"]["raw_path"])


## 3. Data Loading & Validation

The loader checks for the real Kaggle CSV at `data/raw/Mall_Customers.csv`. If it is absent, a realistic synthetic dataset is generated and saved there so the pipeline is runnable immediately. Drop in the real CSV and re-run — the rest of the pipeline is unaffected.

Schema validation enforces:
- All expected columns present  
- No missing values in numeric columns  
- No duplicate `CustomerID` values  
- No negative numeric values

In [ ]:
logger = get_logger(
    "notebook",
    log_dir=config["paths"]["log_dir"],
    log_filename="notebook.log",
    level=config["logging"]["level"],
)

df = load_raw_data(config, logger)
print(f"Shape: {df.shape}")
df.head(10)


In [ ]:
print("--- Data Types ---")
print(df.dtypes)
print()
print("--- Missing Values ---")
print(df.isnull().sum())
print()
print("--- Descriptive Statistics ---")
df.describe().round(2)


## 4. Exploratory Data Analysis

Before clustering, we inspect the marginal distributions of each feature and their pairwise relationships. This informs our confidence in k=5 and surfaces any data quality issues not caught by schema validation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

for ax, feat, color in zip(axes, features, colors):
    ax.hist(df[feat], bins=20, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(df[feat].mean(), color="black", linestyle="--", linewidth=1.2, label=f"mean={df[feat].mean():.1f}")
    ax.set_title(feat)
    ax.legend(fontsize=9)

fig.suptitle("Feature Distributions", fontsize=13, y=1.02)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_feature_distributions.png",
            dpi=130, bbox_inches="tight")
plt.show()
print("Saved: outputs/plots/nb_feature_distributions.png")


In [ ]:
# Gender breakdown
print("Gender distribution:")
print(df["Gender"].value_counts())
print()

# Correlation matrix for numeric features
print("Pearson correlations:")
df[features].corr().round(3)


In [ ]:
# Annual Income vs Spending Score — the classic 2D view used in most tutorials
fig, ax = plt.subplots(figsize=(7, 5))
male = df[df["Gender"] == "Male"]
female = df[df["Gender"] == "Female"]
ax.scatter(male["Annual Income (k$)"], male["Spending Score (1-100)"],
           label="Male", alpha=0.7, s=50, color="#4C72B0")
ax.scatter(female["Annual Income (k$)"], female["Spending Score (1-100)"],
           label="Female", alpha=0.7, s=50, color="#C44E52")
ax.set_xlabel("Annual Income (k$)")
ax.set_ylabel("Spending Score (1-100)")
ax.set_title("Income vs Spending Score — by Gender (pre-clustering)")
ax.legend()
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_income_vs_spend_gender.png",
            dpi=130, bbox_inches="tight")
plt.show()


## 5. Feature Engineering

K-means is a distance-based algorithm — it is sensitive to feature scale. A customer with `Annual Income = 100` and `Age = 25` would have the income dimension dominate the Euclidean distance calculation without scaling.

`FeatureEngineer` wraps `StandardScaler` with explicit `fit_transform` / `transform` separation:
- **`fit_transform(df)`** — learns μ and σ from the data, then scales it.  
- **`transform(new_df)`** — applies the *same* μ/σ learned above. Never refits.

This prevents train/test leakage and ensures new customers can be scored using exactly the same transformation used at training time.

In [ ]:
feat_cfg = config["features"]
fe = FeatureEngineer(
    feature_columns=feat_cfg["clustering_features"],
    scale=feat_cfg["scale_features"],
)
X_scaled = fe.fit_transform(df, logger)

print(f"Feature matrix shape : {X_scaled.shape}")
print(f"Post-scaling mean    : {X_scaled.mean(axis=0).round(4)}")
print(f"Post-scaling std     : {X_scaled.std(axis=0).round(4)}")
print()
print("Scaler parameters learned:")
for fname, mu, sig in zip(feat_cfg["clustering_features"], fe.scaler.mean_, fe.scaler.scale_):
    print(f"  {fname:30s}  mean={mu:.2f}  std={sig:.2f}")


## 6. Model Training

We use `k-means++` initialisation (Arthur & Vassilvitskii, 2007) which seeds centroids to be spread out — substantially better than random initialisation and reduces the chance of landing in a poor local minimum.

Hyperparameters (from `config.yaml`):
| Parameter | Value | Reason |
|-----------|-------|--------|
| `n_clusters` | 5 | Standard choice for Mall Customers; elbow confirms it |
| `init` | `k-means++` | Superior convergence vs random |
| `n_init` | 10 | Runs 10 different initialisations, keeps best inertia |
| `max_iter` | 300 | Upper bound on EM iterations per run |
| `random_state` | 42 | Full reproducibility |

In [ ]:
model_cfg = config["model"]
model = CustomerSegmentationModel(
    n_clusters=model_cfg["n_clusters"],
    init=model_cfg["init"],
    n_init=model_cfg["n_init"],
    max_iter=model_cfg["max_iter"],
    random_state=model_cfg["random_state"],
)
model.fit(X_scaled, logger)

print(f"Converged in  : {model.model.n_iter_} iterations")
print(f"Inertia (WCSS): {model.inertia_:.4f}")
print(f"Cluster sizes : {dict(zip(*np.unique(model.labels_, return_counts=True)))}")


## 7. Cluster Evaluation

### 7.1 Elbow Method (informational)
Inertia (within-cluster sum of squares) drops steeply up to k=5 and then flattens — validating our fixed choice.

### 7.2 Silhouette Score
Measures how similar each point is to its own cluster versus the nearest neighbouring cluster.  
- Range: **−1** (wrong cluster) → **+1** (perfect separation)  
- Rule of thumb: > 0.5 is good; > 0.7 is strong.

### 7.3 Davies-Bouldin Index
Measures the average ratio of within-cluster scatter to between-cluster distance.  
- Range: **≥ 0**; lower is better.

In [ ]:
# Elbow curve
eval_cfg = config["evaluation"]
inertias = compute_elbow_curve(
    X_scaled,
    k_range=tuple(eval_cfg["elbow_k_range"]),
    random_state=model_cfg["random_state"],
    logger=logger,
)

fig, ax = plt.subplots(figsize=(8, 4))
ks = list(inertias.keys())
vals = list(inertias.values())
ax.plot(ks, vals, marker="o", linewidth=2, color="#4C72B0")
ax.axvline(5, color="#C44E52", linestyle="--", linewidth=1.5, label="Chosen k=5")
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia (WCSS)")
ax.set_title("Elbow Method")
ax.set_xticks(ks)
ax.legend()
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_elbow.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
sil_score = compute_silhouette(X_scaled, model.labels_)
db_score  = compute_davies_bouldin(X_scaled, model.labels_)

print(f"Silhouette Score      : {sil_score:.4f}  (higher is better; range -1 to +1)")
print(f"Davies-Bouldin Index  : {db_score:.4f}  (lower is better; range ≥ 0)")
print(f"Inertia               : {model.inertia_:.4f}")


## 8. Visualisations

We generate four plots — the same ones that `run_pipeline.py` saves automatically.  
All plots are rendered inline here AND saved to `outputs/plots/`.

In [ ]:
# 8.1 — Cluster size bar chart
labels = model.labels_
unique, counts = np.unique(labels, return_counts=True)
colors = sns.color_palette("viridis", n_colors=len(unique))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(u) for u in unique], counts, color=colors)
for i, c in enumerate(counts):
    ax.text(i, c + max(counts)*0.01, str(c), ha="center", fontweight="bold")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of Customers")
ax.set_title("Customer Count per Cluster")
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_cluster_sizes.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# 8.2 — 2D scatter: Annual Income vs Spending Score
plot_df = df.copy()
plot_df["Cluster"] = labels.astype(str)

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=plot_df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette="viridis",
    s=70, alpha=0.85, ax=ax,
)
ax.set_title("Customer Segments: Income vs Spending Score")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_scatter_income_spend.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# 8.3 — 2D scatter: Age vs Spending Score
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=plot_df,
    x="Age",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette="viridis",
    s=70, alpha=0.85, ax=ax,
)
ax.set_title("Customer Segments: Age vs Spending Score")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_scatter_age_spend.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# 8.4 — 3D scatter (all three clustering features)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(9, 7))
ax3d = fig.add_subplot(111, projection="3d")
scatter = ax3d.scatter(
    df["Age"], df["Annual Income (k$)"], df["Spending Score (1-100)"],
    c=labels, cmap="viridis", s=50, alpha=0.85,
)
ax3d.set_xlabel("Age")
ax3d.set_ylabel("Annual Income (k$)")
ax3d.set_zlabel("Spending Score (1-100)")
ax3d.set_title("3D Customer Segmentation View")
legend = ax3d.legend(*scatter.legend_elements(), title="Cluster", loc="upper left")
ax3d.add_artist(legend)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_cluster_3d.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# 8.5 — Pairplot across all three features
pair_df = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].copy()
pair_df["Cluster"] = labels.astype(str)

grid = sns.pairplot(pair_df, hue="Cluster", palette="viridis",
                    diag_kind="kde", plot_kws={"alpha": 0.7, "s": 40})
grid.fig.suptitle("Pairplot Across All Clustering Features", y=1.02)
grid.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_pairplot.png", dpi=130, bbox_inches="tight")
plt.show()


## 9. Business Interpretation

The cluster profiles below translate raw cluster IDs into actionable business archetypes. These are computed from the mean feature values and gender split of each cluster.

In [ ]:
profiles = build_cluster_profiles(
    df, labels,
    feature_columns=feat_cfg["clustering_features"],
    logger=logger,
)

rows = []
archetype_names = {
    # These are heuristic labels — inspect your own run's mean values and
    # rename accordingly. The pipeline assigns numeric IDs; business meaning
    # is added here after inspection.
}

for cid, p in profiles.items():
    rows.append({
        "Cluster": cid.replace("cluster_", ""),
        "Size": p["size"],
        "%": f"{p['pct_of_total']}%",
        "Mean Age": p["mean_Age"],
        "Mean Income (k$)": p["mean_Annual Income (k$)"],
        "Mean Spend Score": p["mean_Spending Score (1-100)"],
        "Female %": p["gender_pct"].get("Female", 0),
    })

profile_df = pd.DataFrame(rows).set_index("Cluster")
print(profile_df.to_string())


In [ ]:
# Radar / spider chart — shows each cluster's character across the three dimensions
from matplotlib.patches import FancyArrowPatch

fig, axes = plt.subplots(1, 5, figsize=(15, 4), subplot_kw=dict(polar=False))
feature_labels = ["Mean Age", "Mean Income (k$)", "Mean Spend Score"]
palette = sns.color_palette("viridis", n_colors=5)

for ax, (_, row), color in zip(axes, profile_df.iterrows(), palette):
    vals = [row["Mean Age"], row["Mean Income (k$)"], row["Mean Spend Score"]]
    ax.barh(feature_labels, vals, color=color, alpha=0.85)
    ax.set_title(f"Cluster {row.name}", fontweight="bold")
    ax.set_xlim(0, max(profile_df["Mean Income (k$)"].max(), 100) * 1.1)
    ax.tick_params(labelsize=8)

fig.suptitle("Cluster Profiles — Mean Feature Values", fontsize=12)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "plots" / "nb_cluster_profiles.png", dpi=130, bbox_inches="tight")
plt.show()


### Interpreting the Clusters

After inspecting the mean values, typical cluster archetypes for this dataset are:

| Cluster | Archetype | Description | Strategy |
|---------|-----------|-------------|----------|
| High Income · High Spend | **Target Customers** | Already buying premium — retain & upsell | Loyalty rewards, early access |
| High Income · Low Spend | **Cautious Savers** | Have money but don't spend it here | Targeted promotions, trust-building |
| Low Income · High Spend | **Impulsive Shoppers** | Young, spend freely despite modest income | Instalment options, trendy products |
| Low Income · Low Spend | **Budget Shoppers** | Older, cost-conscious | Discount campaigns, value bundles |
| Average Everything | **Standard Customers** | Mid-range across all dimensions | General promotions |

> **Note:** The actual cluster IDs (0–4) that map to each archetype depend on your run's initialisation. Read the profile table above and apply the descriptions accordingly.


## 10. Saving Artifacts

Everything is already saved when `run_pipeline.py` runs. Here we explicitly show what was produced and add the cluster assignments back to the original dataframe for export.

In [ ]:
from src.pipeline.persistence import save_artifact, save_json

# Show what already exists from the pipeline run
print("=== Output Artifacts ===")
for p in sorted(Path(config["paths"]["model_dir"]).iterdir()):
    print(f"  [model]   {p.name}  ({p.stat().st_size:,} bytes)")
for p in sorted(Path(config["paths"]["metrics_dir"]).iterdir()):
    print(f"  [metrics] {p.name}  ({p.stat().st_size:,} bytes)")
for p in sorted(Path(config["paths"]["plots_dir"]).iterdir()):
    print(f"  [plots]   {p.name}  ({p.stat().st_size:,} bytes)")


In [ ]:
# Final labeled dataset — customers with their cluster assignment
labeled_df = df.copy()
labeled_df["Cluster"] = labels
print(f"Labeled dataset shape: {labeled_df.shape}")
print(f"\nCluster value counts:\n{labeled_df['Cluster'].value_counts().sort_index()}")
labeled_df.head(10)


In [ ]:
# Save labeled dataset (pipeline already does this; shown explicitly for clarity)
out_path = Path(config["data"]["processed_path"])
labeled_df.to_csv(out_path, index=False)
print(f"Labeled dataset saved to: {out_path}")
print()
print("=== Pipeline complete ===")
print(f"Silhouette Score    : {sil_score:.4f}")
print(f"Davies-Bouldin Index: {db_score:.4f}")
print(f"Inertia             : {model.inertia_:.4f}")
print(f"Total customers     : {len(df)}")
print(f"Clusters            : {model.n_clusters}")


---
## Next Steps

1. **Replace synthetic data** — drop the real `Mall_Customers.csv` from Kaggle into `data/raw/` and re-run `python scripts/run_pipeline.py`. No code changes needed.
2. **Tune k** — change `model.n_clusters` in `config/config.yaml` and re-run.
3. **Add features** — extend `features.clustering_features` in `config/config.yaml` to include `Gender` (one-hot encoded) or other engineered features.
4. **Score new customers** — load `outputs/models/kmeans_model.joblib` and `outputs/models/scaler.joblib` with `joblib.load()`, call `scaler.transform(new_data)`, then `model.predict(X_scaled)` to assign any new customer to an existing cluster.
5. **Run the full test suite** — in any environment with pip: `pip install pytest && pytest tests/ -v`
